In [1]:
import torch
import numpy as np


class WeaklyConvexProblem:
    """
    Defines the objective function phi(x) = f(x) + r(x).
    Based on Example 2.1 (Phase Retrieval) or simple |x^2 - 1|.
    f_objective: The stochastic loss function f(x)
    """

    def __init__(self, f_objective, r_objective, model_gen, rho=2.0):
        self.f = f_objective
        self.r = r_objective
        self.get_model = model_gen
        self.rho = rho  # Used to guide the Solver's beta_t

    def objective(self, x, data_batch=None):
        return self.f(x, data_batch) + self.r(x)

    def solve_exact(self, x_t, batch, beta_t):
        """
        Optional: Implement the exact argmin for Algorithm 4.1.
        Returns None if no closed-form solution is available.
        """
        return None

In [18]:
class ModelBasedSolver:
    """
    Implements Algorithm 1.2: Stochastic Model-Based Minimization.
    """
    def __init__(self, problem, data, x_init, T, batch_size=1, gamma=0.1):
        self.prob = problem
        self.data = data # The population P
        self.x = x_init.clone().detach()
        self.T = T
        self.gamma = gamma # Step-size parameter
        self.batch_size = batch_size
        self.history = []
        
    def get_beta(self, t):
        # The paper requires beta_t > rho 
        # We add a small constant (e.g., 0.1) to ensure strict strong convexity
        rho_hat = self.prob.rho + 0.1 
        return rho_hat + (1.0 / self.gamma) * np.sqrt(t + 1.0)

    def run(self):
        
        for t in range(self.T):            
            beta_t = self.get_beta(t)
            
            indices = torch.randint(0, len(self.data), (self.batch_size,))
            batch = self.data[indices]
            
            # Check for closed-form solution first
            exact_x = self.prob.solve_exact(self.x, batch, beta_t)
            
            if exact_x is not None:
                self.x = exact_x.detach().clone()
            else:
                # Fallback to your LBFGS numerical solver
                x_t = self.x.clone().detach()       
                g_xt = self.prob.get_model(x_t, batch)
            
                # SUBPROBLEM: x_{t+1} = argmin { f_xt(y) + (1/2*alpha) * ||y-x_t||^2 }
                y = x_t.clone().requires_grad_(True)
                inner_opt = torch.optim.LBFGS([y], lr=1, max_iter=20)
    
                def closure():                
                    inner_opt.zero_grad()
                    model_val = g_xt(y) + self.prob.r(y) # Add the regularizer r(y)
                    penalty = (beta_t / 2) * torch.norm(y - x_t)**2
                    loss = model_val + penalty
                    loss.backward()
                    return loss
    
                inner_opt.step(closure)
                self.x = y.detach().clone()
            
            if t % 50 == 0:
                # Calculate the True Objective (over the whole population)
                # This represents the 'Population Risk'
                with torch.no_grad():
                    mean_f = torch.mean(self.prob.f(self.x, self.data))
                    reg_val = self.prob.r(self.x)
                    total_phi = mean_f + reg_val
                    
                x_norm = torch.norm(self.x).item()
                print(f"Iter {t:4}: x = {x_norm:.4f} | True phi(x) = {total_phi.item():.4f}")

        return self.x

In [5]:
def f_stochastic(x, batch):
        # |<a, x>^2 - b|
        a, b = batch[:, :-1], batch[:, -1]
        return torch.abs((x @ a.T)**2 - b)

class PhaseRetrievalProblem(WeaklyConvexProblem):
    def __init__(self, rho=2.0):
        # We define a dummy model_gen because solve_exact will override it
        super().__init__(f_objective=f_stochastic, r_objective=lambda x: 0, 
                         model_gen=None, rho=rho)

    def solve_exact(self, x_t, batch, beta_t):
        """
        Closed-form solution for Stochastic Prox-Linear Phase Retrieval.
        Based on Equation 5.2 in Davis & Drusvyatskiy (2018).
        """
        # batch represents (a_i, b_i)
        # For simplicity, let's assume batch is a single sample (a, b)
        a = batch[0, :-1]
        b = batch[0, -1]
        
        # Current residual and gradient of the inner quadratic
        dot_prod = torch.dot(a, x_t)
        gamma = dot_prod**2 - b
        zeta = 2 * dot_prod * a
        
        # Step size lambda in the paper is 1/beta_t
        # The formula: Delta = proj_[-1, 1](-gamma / (lambda * ||zeta||^2)) * lambda * zeta
        # Simplified for our beta_t notation:
        norm_zeta_sq = torch.norm(zeta)**2
        
        if norm_zeta_sq < 1e-9:
            return x_t
            
        scale = torch.clamp(-gamma * beta_t / norm_zeta_sq, min=-1.0, max=1.0)
        delta = (scale / beta_t) * zeta
        
        return x_t + delta

In [16]:
# Setup Synthetic Data for Phase Retrieval
torch.manual_seed(42)
d = 10
m = 100
true_x = torch.randn(d)
a_matrix = torch.randn(m, d)
b_vector = (a_matrix @ true_x)**2 

# Combine into a population: each row is [a_1, ..., a_d, b]
population_data = torch.cat([a_matrix, b_vector.unsqueeze(1)], dim=1)

# Initialize Problem and Solver
prob = PhaseRetrievalProblem(rho=2.0)
solver = ModelBasedSolver(
    problem=prob, 
    data=population_data, 
    x_init=torch.randn(d), 
    T=500, 
    gamma=0.1
)

final_x = solver.run()
print(f"Final sampled x: {torch.norm(final_x).item():.4f}")
print(f"Final reconstruction error: {torch.norm(final_x - true_x).item():.4f}")

Iter    0: x = 3.4600 | True phi(x) = 9.0899
Iter   50: x = 3.0227 | True phi(x) = 4.0209
Iter  100: x = 2.7715 | True phi(x) = 1.6202
Iter  150: x = 2.6419 | True phi(x) = 0.4002
Iter  200: x = 2.6672 | True phi(x) = 0.0422
Iter  250: x = 2.6660 | True phi(x) = 0.0110
Iter  300: x = 2.6648 | True phi(x) = 0.0003
Iter  350: x = 2.6648 | True phi(x) = 0.0000
Iter  400: x = 2.6648 | True phi(x) = 0.0000
Iter  450: x = 2.6648 | True phi(x) = 0.0000
Final sampled x: 2.6648
Final reconstruction error: 0.0000


In [21]:
def max_parabola_phi(x, batch):
    """The 'True' Objective: max(x^2, (x-4)^2)"""
    f1 = (x - batch[:, 0])**2
    f2 = (x - batch[:, 1])**2
    return torch.max(f1, f2)


def max_parabola_model_gen(x_t, batch):
    """
    Returns the Stochastic Model for the Pointwise Max.
    Based on Example 2.6 and Section 4.2[cite: 984].
    """
    # Detach constants for the inner loop
    x_t_val = x_t.detach()
    f1_vals = (x_t_val - batch[:, 0])**2
    f2_vals = (x_t_val - batch[:, 1])**2

    g1_vals = 2 * (x_t_val - batch[:, 0])
    g2_vals = 2 * (x_t_val - batch[:, 1])

    def model(y):
        # Linearize components for each sample in batch, then average
        l1 = f1_vals + g1_vals * (y - x_t_val)
        l2 = f2_vals + g2_vals * (y - x_t_val)
        return torch.mean(torch.max(l1, l2))
    
    return model

def elastic_net_regularizer(x, l1_ratio=0.5, mu=0.01):
    """
    Combines L1 and L2 regularization. 
    Matches the 'regularized population risk' framework.
    """
    l1_term = l1_ratio * torch.norm(x, p=1)
    l2_term = (1 - l1_ratio) * 0.5 * torch.norm(x, p=2)**2
    # Ensure the result is a tensor attached to the same device as x
    return mu * (l1_term + l2_term)
# Create a population of 1000 data points (targets for the parabolas)
# Center them around 0 and 4 so the "average" intersection is near 2
torch.manual_seed(42)
population_data = torch.randn(1000, 2)
population_data[:, 0] += 0.0  # Centers for f1
population_data[:, 1] += 4.0  # Centers for f2
population_data[:, 1] += 4.0  # Centers for f2
# 1. Setup Problem
# rho=2 because the components (x-xi)^2 have a second derivative of 2
prob = WeaklyConvexProblem(
    f_objective=max_parabola_phi,
    r_objective=elastic_net_regularizer,
    model_gen=max_parabola_model_gen,
    rho=2.0
)

# 2. Run with Mini-Batching
solver = ModelBasedSolver(
    problem=prob,
    data=population_data,
    x_init=torch.tensor([10.0]),
    T=1000,
    batch_size=32,
    gamma=0.5
)

final_x = solver.run()
print(f"Final sampled x: {final_x.item():.4f}")

Iter    0: x = 5.5768 | True phi(x) = 32.3955
Iter   50: x = 3.9014 | True phi(x) = 21.8445
Iter  100: x = 3.9698 | True phi(x) = 21.7957
Iter  150: x = 3.9300 | True phi(x) = 21.8188
Iter  200: x = 3.9974 | True phi(x) = 21.7908
Iter  250: x = 4.0221 | True phi(x) = 21.7947
Iter  300: x = 4.1257 | True phi(x) = 21.8835
Iter  350: x = 4.0212 | True phi(x) = 21.7945
Iter  400: x = 3.9385 | True phi(x) = 21.8126
Iter  450: x = 3.9587 | True phi(x) = 21.8005
Iter  500: x = 3.9598 | True phi(x) = 21.7999
Iter  550: x = 3.9833 | True phi(x) = 21.7922
Iter  600: x = 4.0877 | True phi(x) = 21.8356
Iter  650: x = 3.9803 | True phi(x) = 21.7928
Iter  700: x = 3.9969 | True phi(x) = 21.7908
Iter  750: x = 4.0624 | True phi(x) = 21.8138
Iter  800: x = 3.9681 | True phi(x) = 21.7963
Iter  850: x = 4.0355 | True phi(x) = 21.7989
Iter  900: x = 3.9991 | True phi(x) = 21.7909
Iter  950: x = 3.9165 | True phi(x) = 21.8300
Final sampled x: 4.0274
